## Validación del módulo de preprocesamiento

En este notebook se comprueba el correcto funcionamiento del archivo `src/preprocessing.py`, encargado de automatizar el preprocesamiento de los tres datasets del proyecto (`adults`, `combined` y `toddlers`).

El objetivo de esta validación es verificar que:
- los datasets se cargan correctamente,
- las columnas objetivo se codifican a formato binario,
- se eliminan las variables no aptas para modelado,
- se aplica correctamente la transformación de variables numéricas y categóricas,
- se generan los conjuntos de train y test,
- y los datos resultantes quedan sin valores nulos y listos para la fase de modelado.

In [3]:
import sys
from pathlib import Path

# Ruta raíz del proyecto
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

# Carpetas de datos
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Importar el módulo completo para evitar problemas de import en notebook
import src.preprocessing as prep

In [4]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: C:\Users\laura\Desktop\tesTEA
RAW_DIR: C:\Users\laura\Desktop\tesTEA\data\raw
PROCESSED_DIR: C:\Users\laura\Desktop\tesTEA\data\processed


## 1. Validación del dataset `adults`

En este bloque se comprueba que el dataset `adults`:
- se carga correctamente,
- contiene las columnas esperadas,
- puede pasar por el pipeline de preprocesamiento,
- y genera conjuntos `train` y `test` sin valores nulos.

In [5]:
ADULTS_PATH = RAW_DIR / "autism_screening.csv"

adults_df = prep.load_dataset(ADULTS_PATH)

print("Shape adults:", adults_df.shape)
print("\nColumnas adults:")
print(adults_df.columns.tolist())

adults_df.head()

Shape adults: (704, 21)

Columnas adults:
['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'age', 'gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'used_app_before', 'result', 'age_desc', 'relation', 'Class/ASD']


,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,gender,ethnicity,jundice,austim,contry_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,1,1,0,0,1,1,0,0,...,f,White-European,no,no,United States,no,6.0,18 and more,Self,NO
1,1,1,0,1,0,0,0,1,0,1,...,m,Latino,no,yes,Brazil,no,5.0,18 and more,Self,NO
2,1,1,0,1,1,0,1,1,1,1,...,m,Latino,yes,yes,Spain,no,8.0,18 and more,Parent,YES
3,1,1,0,1,0,0,1,1,0,1,...,f,White-European,no,yes,United States,no,6.0,18 and more,Self,NO
4,1,0,0,0,0,0,0,1,0,0,...,f,?,no,no,Egypt,no,2.0,18 and more,?,NO


In [6]:
adults_train, adults_test, adults_preprocessor = prep.preprocess_and_save_dataset(
    df=adults_df,
    dataset_name="adults",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\laura\Desktop\tesTEA\data\processed\adults_train.csv
Guardado: C:\Users\laura\Desktop\tesTEA\data\processed\adults_test.csv


In [7]:
print("Shape adults_train:", adults_train.shape)
print("Shape adults_test:", adults_test.shape)

print("\nNulos en adults_train:", adults_train.isna().sum().sum())
print("Nulos en adults_test:", adults_test.isna().sum().sum())

print("\nPrimeras columnas de adults_train:")
print(adults_train.columns.tolist()[:15])

print("\nDistribución del target en train:")
print(adults_train["target"].value_counts(dropna=False))

print("\nDistribución del target en test:")
print(adults_test["target"].value_counts(dropna=False))

Shape adults_train: (563, 100)
Shape adults_test: (141, 100)

Nulos en adults_train: 0
Nulos en adults_test: 0

Primeras columnas de adults_train:
['num__age', 'cat__gender_f', 'cat__gender_m', 'cat__ethnicity_Asian', 'cat__ethnicity_Black', 'cat__ethnicity_Hispanic', 'cat__ethnicity_Latino', 'cat__ethnicity_Middle Eastern ', 'cat__ethnicity_Others', 'cat__ethnicity_Pasifika', 'cat__ethnicity_South Asian', 'cat__ethnicity_Turkish', 'cat__ethnicity_White-European', 'cat__ethnicity_others', 'cat__ethnicity_unknown']

Distribución del target en train:
target
0    412
1    151
Name: count, dtype: int64

Distribución del target en test:
target
0    103
1     38
Name: count, dtype: int64


## 2. Validación del dataset `combined`

En este bloque se valida el funcionamiento del preprocesado para el dataset `combined`, comprobando:
- carga del dataset,
- correspondencia de columnas con la configuración,
- generación de train/test,
- ausencia de nulos tras el pipeline.

In [8]:
COMBINED_PATH = RAW_DIR / "Autism_Screening_Data_Combined.csv"

combined_df = prep.load_dataset(COMBINED_PATH)

print("Shape combined:", combined_df.shape)
print("\nColumnas combined:")
print(combined_df.columns.tolist())

combined_df.head()

Shape combined: (6075, 15)

Columnas combined:
['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Age', 'Sex', 'Jauundice', 'Family_ASD', 'Class']


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age,Sex,Jauundice,Family_ASD,Class
0,1,1,0,1,0,0,1,1,0,0,15,m,no,no,NO
1,0,1,1,1,0,1,1,0,1,0,15,m,no,no,NO
2,1,1,1,0,1,1,1,1,1,1,15,f,no,yes,YES
3,1,1,1,1,1,1,1,1,0,0,16,f,no,no,YES
4,1,1,1,1,1,1,1,1,1,1,15,f,no,no,YES


In [9]:
combined_train, combined_test, combined_preprocessor = prep.preprocess_and_save_dataset(
    df=combined_df,
    dataset_name="combined",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\laura\Desktop\tesTEA\data\processed\combined_train.csv
Guardado: C:\Users\laura\Desktop\tesTEA\data\processed\combined_test.csv


In [12]:
print("Shape combined_train:", combined_train.shape)
print("Shape combined_test:", combined_test.shape)

print("\nNulos en combined_train:", combined_train.isna().sum().sum())
print("Nulos en combined_test:", combined_test.isna().sum().sum())

print("\nPrimeras columnas de combined_train:")
print(combined_train.columns.tolist()[:15])

print("\nDistribución del target en train:")
print(combined_train["target"].value_counts(dropna=False))

print("\nDistribución del target en test:")
print(combined_test["target"].value_counts(dropna=False))


Shape combined_train: (4860, 18)
Shape combined_test: (1215, 18)

Nulos en combined_train: 0
Nulos en combined_test: 0

Primeras columnas de combined_train:
['num__Age', 'cat__Sex_f', 'cat__Sex_m', 'cat__Jauundice_no', 'cat__Jauundice_yes', 'cat__Family_ASD_no', 'cat__Family_ASD_yes', 'aq__A1', 'aq__A2', 'aq__A3', 'aq__A4', 'aq__A5', 'aq__A6', 'aq__A7', 'aq__A8']

Distribución del target en train:
target
0    3417
1    1443
Name: count, dtype: int64

Distribución del target en test:
target
0    854
1    361
Name: count, dtype: int64


## 3. Validación del dataset `toddlers`

En este bloque se valida el preprocesamiento del dataset `toddlers`, que presenta algunas particularidades en nombres de columnas (por ejemplo, espacios en el target o categorías con formatos distintos).

Se comprueba igualmente:
- la carga correcta,
- la ejecución del pipeline,
- la generación de train/test,
- y la ausencia de nulos en el resultado final.

In [13]:
TODDLERS_PATH = RAW_DIR / "Toddler Autism dataset July 2018.csv"

toddlers_df = prep.load_dataset(TODDLERS_PATH)

print("Shape toddlers:", toddlers_df.shape)
print("\nColumnas toddlers:")
print(toddlers_df.columns.tolist())

toddlers_df.head()

Shape toddlers: (1054, 19)

Columnas toddlers:
['Case_No', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Age_Mons', 'Qchat-10-Score', 'Sex', 'Ethnicity', 'Jaundice', 'Family_mem_with_ASD', 'Who completed the test', 'Class/ASD Traits ']


,Case_No,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age_Mons,Qchat-10-Score,Sex,Ethnicity,Jaundice,Family_mem_with_ASD,Who completed the test,Class/ASD Traits
0,1,0,0,0,0,0,0,1,1,0,1,28,3,f,middle eastern,yes,no,family member,No
1,2,1,1,0,0,0,1,1,0,0,0,36,4,m,White European,yes,no,family member,Yes
2,3,1,0,0,0,0,0,1,1,0,1,36,4,m,middle eastern,yes,no,family member,Yes
3,4,1,1,1,1,1,1,1,1,1,1,24,10,m,Hispanic,no,no,family member,Yes
4,5,1,1,0,1,1,1,1,1,1,1,20,9,f,White European,no,yes,family member,Yes


In [14]:
toddlers_train, toddlers_test, toddlers_preprocessor = prep.preprocess_and_save_dataset(
    df=toddlers_df,
    dataset_name="toddlers",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\laura\Desktop\tesTEA\data\processed\toddlers_train.csv
Guardado: C:\Users\laura\Desktop\tesTEA\data\processed\toddlers_test.csv


In [15]:
print("Shape toddlers_train:", toddlers_train.shape)
print("Shape toddlers_test:", toddlers_test.shape)

print("\nNulos en toddlers_train:", toddlers_train.isna().sum().sum())
print("Nulos en toddlers_test:", toddlers_test.isna().sum().sum())

print("\nPrimeras columnas de toddlers_train:")
print(toddlers_train.columns.tolist()[:20])

print("\nDistribución del target en train:")
print(toddlers_train["target"].value_counts(dropna=False))

print("\nDistribución del target en test:")
print(toddlers_test["target"].value_counts(dropna=False))

Shape toddlers_train: (843, 34)
Shape toddlers_test: (211, 34)

Nulos en toddlers_train: 0
Nulos en toddlers_test: 0

Primeras columnas de toddlers_train:
['num__Age_Mons', 'cat__Sex_f', 'cat__Sex_m', 'cat__Ethnicity_Hispanic', 'cat__Ethnicity_Latino', 'cat__Ethnicity_Native Indian', 'cat__Ethnicity_Others', 'cat__Ethnicity_Pacifica', 'cat__Ethnicity_White European', 'cat__Ethnicity_asian', 'cat__Ethnicity_black', 'cat__Ethnicity_middle eastern', 'cat__Ethnicity_mixed', 'cat__Ethnicity_south asian', 'cat__Jaundice_no', 'cat__Jaundice_yes', 'cat__Family_mem_with_ASD_no', 'cat__Family_mem_with_ASD_yes', 'cat__Who completed the test_Health Care Professional', 'cat__Who completed the test_Health care professional']

Distribución del target en train:
target
1    582
0    261
Name: count, dtype: int64

Distribución del target en test:
target
1    146
0     65
Name: count, dtype: int64


## 4. Resumen de la validación

Tras ejecutar el notebook se comprueba que:

- `src/preprocessing.py` funciona correctamente con los tres datasets del proyecto.
- Se generan conjuntos de entrenamiento y test para `adults`, `combined` y `toddlers`.
- Las transformaciones de variables numéricas, categóricas y AQ se aplican correctamente.
- El target se codifica a formato binario.
- Los datasets resultantes no contienen valores nulos.
- Los archivos procesados se guardan en `data/processed`.

Este notebook actúa como comprobación funcional previa al notebook definitivo de preprocesamiento del proyecto.

In [16]:
print("Resumen validación preprocessing")
print("- Adults procesado correctamente")
print("- Combined procesado correctamente")
print("- Toddlers procesado correctamente")
print("- Todos los datasets quedan sin nulos")
print("- Los targets se codifican en formato binario (0/1)")
print("- Los CSV procesados se guardan en data/processed")

Resumen validación preprocessing
- Adults procesado correctamente
- Combined procesado correctamente
- Toddlers procesado correctamente
- Todos los datasets quedan sin nulos
- Los targets se codifican en formato binario (0/1)
- Los CSV procesados se guardan en data/processed
